# Q4 — Machine Translation (Multi30k EN→DE)
**Models:** Seq2Seq+Attention (GPU, trained) → Helsinki-NLP opus-mt (GPU, pretrained)

**Dataset:** Full Multi30k (~29K train)

**Runtime:** T4 GPU required

## 1. Environment Setup

In [ ]:
import os, subprocess, sys

from google.colab import drive
drive.mount('/content/drive')

DRIVE_OUTPUT = '/content/drive/MyDrive/467-takehome-outputs/q4'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)
print('Drive mounted, output dir ready.')

In [ ]:
REPO_DIR = '/content/467-takehome'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/zubeyralmaho/467-takehome.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

## 2. Run Seq2Seq+Attention (GPU — Training)

Full Multi30k dataset, increased epochs for decent BLEU.

In [ ]:
!python -m src.q4_machine_translation.main \
    --config configs/q4.yaml \
    --final-eval \
    --override \
        models.seq2seq.enabled=true \
        models.transformer.enabled=false \
        models.seq2seq.max_epochs=20 \
        models.seq2seq.early_stopping_patience=5 \
        models.seq2seq.hidden_dim=256 \
        models.seq2seq.embedding_dim=256 \
        models.seq2seq.num_layers=2 \
        models.seq2seq.dropout=0.3 \
        models.seq2seq.batch_size=64 \
        models.seq2seq.learning_rate=0.001 \
        models.seq2seq.teacher_forcing_ratio=0.5 \
        dataset.limit_train_samples=null \
        dataset.limit_validation_samples=null \
        dataset.limit_test_samples=null

In [ ]:
import glob, shutil
latest_run = sorted(glob.glob('outputs/q4/run_*'))[-1]
dest = os.path.join(DRIVE_OUTPUT, os.path.basename(latest_run) + '_seq2seq')
shutil.copytree(latest_run, dest, dirs_exist_ok=True)
print(f'Seq2Seq results saved to Drive: {dest}')

## 3. Run Helsinki-NLP Pretrained Transformer (GPU — Inference)

In [ ]:
!python -m src.q4_machine_translation.main \
    --config configs/q4.yaml \
    --final-eval \
    --override \
        models.seq2seq.enabled=false \
        models.transformer.enabled=true \
        models.transformer.batch_size=16 \
        dataset.limit_train_samples=null \
        dataset.limit_validation_samples=null \
        dataset.limit_test_samples=null

In [ ]:
latest_run = sorted(glob.glob('outputs/q4/run_*'))[-1]
dest = os.path.join(DRIVE_OUTPUT, os.path.basename(latest_run) + '_transformer')
shutil.copytree(latest_run, dest, dirs_exist_ok=True)
print(f'Transformer results saved to Drive: {dest}')

## 4. Results Summary

In [ ]:
import json

print('=== Q4 Results Summary ===')
for run_dir in sorted(glob.glob('outputs/q4/run_*')):
    metrics_file = os.path.join(run_dir, 'metrics.json')
    if os.path.exists(metrics_file):
        with open(metrics_file) as f:
            metrics = json.load(f)
        print(f'\n--- {os.path.basename(run_dir)} ---')
        print(json.dumps(metrics, indent=2, default=str)[:2000])